In [1]:
import route_matrix
import json
import tqdm

/Users/jcrb/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
coordinates = json.load(open("data/tract-centroids.json"))
pairs = json.load(open("data/origin_pairs.json"))

In [3]:
def deduplicate(l):
    new_l = []
    for e in l:
        if not e in new_l:
            new_l.append(e)
    return new_l


origin_coords = {k: coordinates[k] for k, v in pairs.items()}
destination_coords = {k: [coordinates[vi] for vi in deduplicate(v)] 
 for k,v in pairs.items()}

destination_coords

{'Census Tract 814.02': [[-87.6853728497224, 41.92862970854046],
  [-87.61975201545292, 41.89503904573094],
  [-87.64296872542037, 41.86789230931981],
  [-87.60996759660905, 41.89190829618846],
  [-87.62517035037656, 41.834959971716984],
  [-87.63274641803926, 41.88102686254463]],
 'Census Tract 813': [[-87.64262363419903, 41.87924795024878],
  [-87.60996759660905, 41.89190829618846],
  [-87.62623002479897, 41.89255912578125],
  [-87.62621204437693, 41.89915380624777],
  [-87.6428042439965, 41.88529624776741],
  [-87.62093223737456, 41.88504648048074],
  [-87.637795508798, 41.89325552522301]],
 'Census Tract 208.01': [[-87.6716805484159, 41.87866521261404],
  [-87.70368969259569, 41.986891558552614],
  [-87.90389546005812, 41.98026693652026],
  [-87.68102501626824, 41.906771599037214],
  [-87.62093223737456, 41.88504648048074]],
 'Census Tract 2410': [[-87.61975201545292, 41.89503904573094],
  [-87.70071234755646, 41.86075009643528],
  [-87.61736675901997, 41.87772138115119],
  [-87.67

In [4]:
for i, (o, c) in enumerate(origin_coords.items()):
    if i: continue
    lon, lat = c

    print(lat)

41.89297495363672


In [5]:
def get_num_signals(city, nodes_visited):
    def has_signal(node):
        attrs = city.nodes[node]
        if not 'highway' in attrs:
            return False
        elif attrs['highway'] == "traffic_signals":
            return True
        else:
            return False

    num_signals = sum([has_signal[p] for p in nodes_visited])
    return num_signals

In [6]:
total_pairs = 0

for i, (o, c) in tqdm.tqdm(enumerate(origin_coords.items())):
    destinations = [tuple([l[1], l[0]]) for l in destination_coords[o]]

    total_pairs += len(destinations)

print(f"TOTAL COMBINATIONS: {total_pairs}")

664it [00:00, 774046.10it/s]

TOTAL COMBINATIONS: 3796


In [7]:
import datetime as dt
from route_matrix import compute_travel_times, load_api_key
import os
import time

if os.path.exists('./data/real_traffic_paths_traffic_5pm.json'):
    results = json.load(open('./data/real_traffic_paths_traffic_5pm.json', 'r'))
else:
    results = {}
    
for i, (o, c) in tqdm.tqdm(enumerate(origin_coords.items())):
    #print(i, o, c)
    if o in results: continue
    if i == 172: continue

    lon, lat = c

    origin = (lat, lon)
    destinations = [tuple([l[1], l[0]]) for l in destination_coords[o]]
    partial_res = compute_travel_times(origin, destinations,
                     dt.datetime.fromisoformat("2026-11-24T17:00:00-06:00"), load_api_key('key'),
                     routing_preference="TRAFFIC_AWARE", endpoint="routes")

    #print(origin)
    #print(destinations)

    for i, r in enumerate(partial_res):
        r['destination'] = deduplicate(pairs[o])[i]
        r['coords'] = coordinates[r['destination']]

        partial_res[i] = r

    results[o] = partial_res

    json.dump(results, open('./data/real_traffic_paths_traffic_5pm.json', 'w'), indent=2)

664it [04:39,  2.37it/s] 


## Additional time per traffic signal (traffic unaware)

In [8]:
import plotly.graph_objects as go
import numpy as np

signal_dict = json.load(open('data/num_traffic_signals.json', 'r'))
budget_results = json.load(open('results/paths/budget/commutes_old.json'))
results = json.load(open('./data/real_traffic_paths.json', 'r'))

x = []
y = []

for o, r in results.items():
    for d in r:
        dest = d['destination']

        if not o in budget_results: continue
        if not dest in budget_results[o]: continue
        
        raw_result = budget_results[o][dest]
        realworld_result = d['static_duration_s'] 

        unrestricted_path_index = list(raw_result.keys())[0]
        num_signals = signal_dict[o][dest]

        #if num_signals < 10: continue 

        x += [num_signals]
        if not num_signals:
            y += [0]
        else:
            y += [(realworld_result-raw_result[str(unrestricted_path_index)])/num_signals]


rolling_average = [np.mean(np.array(y)[np.array(x) >= t]) for t in range(1, max(x)+1)] 
x_average = np.nan_to_num([np.mean(np.array(y)[np.array(x) == t]) for t in range(1,max(x)+1)], nan=0)

for i, xe in enumerate(x_average):
    if not xe: x_average[i] = x_average[i-1]

fig = go.Figure()
fig.update(**json.load(open('./analysis/graph_config.json', 'r')))
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name="Tract data"))
fig.add_trace(go.Scatter(x=[t for t in range(1, max(x)+1)], y=rolling_average, name="Mean (X >= x)", line_color="#E3170A"))
fig.add_trace(go.Scatter(x=[t for t in range(1, max(x)+1)], y=x_average, name="Mean (X == x)", line_color="#63D471"))

fig.update_yaxes(range=(0, 400), title="Additional time per traffic light")
fig.update_xaxes(range=(-2, 72), title="Number of traffic light encounters")

fig.update_layout(
    font=dict(size=20),                 # global default
    title_font_size=20,
    legend_font_size=16,
    xaxis=dict(title_font_size=20, tickfont_size=20,),
    yaxis=dict(title_font_size=20, tickfont_size=20),
)

fig.show()



mean = np.mean(y)

x = np.array(x)

x = x[x.argsort()[::1]]
y = np.array(y)[x.argsort()[::1]]





print(f"AVERAGE EXTRA TIME PER TRAFFIC LIGHT: {mean}")

/Users/jcrb/Library/Python/3.9/lib/python/site-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/jcrb/Library/Python/3.9/lib/python/site-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


AVERAGE EXTRA TIME PER TRAFFIC LIGHT: 35.262595368581515


In [37]:
fig.write_image("./analysis/figures/traffic_signals.png", scale=3)

## Computing traffic slowdown

In [9]:
def flatten(a, array):
    if not array: return a, None
    [a.append(e) for e in array[0]]
    return a, flatten(a, array[1:])

flatten([], [[1, 2], [3]])[0]

[1, 2, 3]

In [108]:
traffic_data = json.load(open("data/real_traffic_paths_traffic_5pm.json", "r"))

time_increase = {k: {v['destination']:{"delta":v["duration_s"] - v["static_duration_s"], "ratio": v["duration_s"]/(v["static_duration_s"] + 1e-6)} for v in vlist}  for k, vlist in traffic_data.items()}

fig = go.Figure()

clean_data = [f for f in flatten([], [[v['ratio'] for k, v in vlist.items()] for k, vlist in time_increase.items()])[0] if f > 0 and f < 100]
mean = np.mean(clean_data)
fig.add_trace(go.Histogram(x=clean_data))
fig.add_vline(x=mean, line_dash="dash", line_color="red")
fig.update(**json.load(open('./analysis/graph_config.json', 'r')))

fig.update_yaxes(title="Number of occurances")
fig.update_xaxes(title="Traffic time to Free-flow time ratio")

fig.update_layout(font=dict(size=20), title_font_size=20, legend_font_size=16, xaxis=dict(title_font_size=20, tickfont_size=20,), yaxis=dict(title_font_size=20, tickfont_size=20))
fig.show()
fig.write_image("./analysis/figures/traffic_freeflow_histogram.png", scale=3)

In [84]:
import statsmodels
import pandas as pd

path_data = json.load(open('results/paths/initial/commutes-road-breakdown_0s.json', 'r'))

results = []

for origin, v in path_data.items():
    destinations = list(v.keys())
    
    try:
        for d in destinations:
            results.append([origin, d, v[d]['highway_time'], v[d]['city_time'], v[d]['highway_time']/v[d]['length'], v[d]['city_time']/v[d]['length'],  v[d]['signal_penalty']/v[d]['length'], time_increase[origin][d]['delta'], time_increase[origin][d]['ratio']])
    except:
        continue

    

df = pd.DataFrame(results, index=[i for i in range(len(results))], columns=['origin', 'dest', 'highway_time', 'city_time', "highway_ratio", "city_ratio", 'signal_time', "traffic_flow_delta", 'traffic_flow_ratio'])
df['traffic_flow_ratio'] -= 1
df = df.loc[(df.traffic_flow_ratio > -0.5) & (df.traffic_flow_ratio < 3)]
df

,origin,dest,highway_time,city_time,highway_ratio,city_ratio,signal_time,traffic_flow_delta,traffic_flow_ratio
0,Census Tract 8424,Census Tract 3204,759.266383,60.951181,0.925689,0.074311,0.0,347,0.368757
1,Census Tract 8424,Census Tract 8378,824.751626,161.518949,0.836233,0.163767,0.0,802,0.650446
2,Census Tract 8403,Census Tract 812.01,501.591728,223.115378,0.692130,0.307870,0.0,810,0.675000
4,Census Tract 8411,Census Tract 3201,0.000000,368.104454,0.000000,1.000000,0.0,446,0.570332
5,Census Tract 8411,Census Tract 816,0.000000,413.651122,0.000000,1.000000,0.0,596,0.758270
...,...,...,...,...,...,...,...,...,...
3724,Census Tract 2922,Census Tract 8430,0.000000,165.375486,0.000000,1.000000,0.0,36,0.106825
3725,Census Tract 6309,Census Tract 2838,336.188412,540.235171,0.383591,0.616409,0.0,209,0.134926
3726,Census Tract 6309,Census Tract 6505,0.000000,568.170569,0.000000,1.000000,0.0,108,0.121348
3727,Census Tract 6309,Census Tract 712,652.937201,601.270723,0.520597,0.479403,0.0,1106,0.531731


In [101]:
import statsmodels.formula.api as smf

# origin == dest pairs have a static duration of 0, giving a ratio of 0 -- drop them
reg_df = df[df['origin'] != df['dest']]

highway_model = smf.wls('traffic_flow_ratio ~ highway_ratio + 0', data=reg_df).fit(cov_type='HC3', )
city_model = smf.wls('traffic_flow_ratio ~ city_ratio', data=reg_df).fit(cov_type='HC3')
model = smf.wls('traffic_flow_ratio ~ highway_ratio + city_ratio + 0', data=reg_df).fit(cov_type='HC3')
print(model.summary())

                            WLS Regression Results                            
Dep. Variable:     traffic_flow_ratio   R-squared:                       0.325
Model:                            WLS   Adj. R-squared:                  0.325
Method:                 Least Squares   F-statistic:                       nan
Date:                Fri, 06 Nov 2026   Prob (F-statistic):                nan
Time:                        17:35:12   Log-Likelihood:                 762.05
No. Observations:                3604   AIC:                            -1520.
Df Residuals:                    3602   BIC:                            -1508.
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
highway_ratio     0.6283      0.010     63.084

In [102]:
xfit = [0, 1]

fig = go.Figure(go.Scatter(x=df.highway_ratio, y=df.traffic_flow_ratio, mode='markers'))

fig.add_trace(go.Scatter(x=xfit, y=[0, highway_model.params.to_list()[0]*xfit[1]]))
#fig.add_trace(go.Scatter(x=xfit, y=[highway_model.params.to_list()[0], highway_model.params.to_list()[1]*xfit[1]]))

fig.update_xaxes(title="Highway travel time")
fig.update_yaxes(title="Increase in travel time", tickformat=',.0%',)
fig.update(**json.load(open('./analysis/graph_config.json', 'r')))

fig.update_layout(font=dict(size=20), title_font_size=20, legend_font_size=16, xaxis=dict(title_font_size=20, tickfont_size=20,), yaxis=dict(title_font_size=20, tickfont_size=20))
fig.show()

In [110]:
reg_df = df[df['origin'] != df['dest']]

highway_model = smf.wls('traffic_flow_delta ~ highway_time + 0', data=reg_df).fit(cov_type='HC3')
city_model = smf.wls('traffic_flow_delta ~ city_time + 0', data=reg_df).fit(cov_type='HC3')
model = smf.wls('traffic_flow_delta ~ highway_time + city_time + 0', data=reg_df).fit(cov_type='HC3')
print(model.summary())

                                 WLS Regression Results                                
Dep. Variable:     traffic_flow_delta   R-squared (uncentered):                   0.804
Model:                            WLS   Adj. R-squared (uncentered):              0.804
Method:                 Least Squares   F-statistic:                              3889.
Date:                Fri, 06 Nov 2026   Prob (F-statistic):                        0.00
Time:                        17:41:15   Log-Likelihood:                         -24726.
No. Observations:                3604   AIC:                                  4.946e+04
Df Residuals:                    3602   BIC:                                  4.947e+04
Df Model:                           2                                                  
Covariance Type:                  HC3                                                  
                   coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------

In [106]:
xfit = [0, 1500]

fig = go.Figure(go.Scatter(x=df.highway_time, y=df.traffic_flow_delta, mode='markers'))

fig.add_trace(go.Scatter(x=xfit, y=[0, model.params.to_list()[0]*xfit[1]]))
#fig.add_trace(go.Scatter(x=xfit, y=[highway_model.params.to_list()[0], highway_model.params.to_list()[1]*xfit[1]]))

fig.update_xaxes(title="Highway travel time")
fig.update_yaxes(title="Increase in travel time")
fig.update(**json.load(open('./analysis/graph_config.json', 'r')))

fig.update_layout(font=dict(size=20), title_font_size=20, legend_font_size=16, xaxis=dict(title_font_size=20, tickfont_size=20,), yaxis=dict(title_font_size=20, tickfont_size=20))
fig.show()
fig.write_image("./analysis/figures/busy_highway_time.png", scale=3)

fig = go.Figure(go.Scatter(x=df.city_time, y=df.traffic_flow_delta, mode='markers'))

fig.add_trace(go.Scatter(x=xfit, y=[0, model.params.to_list()[0]*xfit[1]]))
#fig.add_trace(go.Scatter(x=xfit, y=[highway_model.params.to_list()[0], highway_model.params.to_list()[1]*xfit[1]]))

fig.update_xaxes(title="City travel time")
fig.update_yaxes(title="Increase in travel time")
fig.update(**json.load(open('./analysis/graph_config.json', 'r')))

fig.update_layout(font=dict(size=20), title_font_size=20, legend_font_size=16, xaxis=dict(title_font_size=20, tickfont_size=20,), yaxis=dict(title_font_size=20, tickfont_size=20))
fig.show()
fig.write_image("./analysis/figures/busy_city_time.png", scale=3)